# Session 32: Logistic Regression Classification

This notebook implements Logistic Regression for at-risk classification.
- 1 = at-risk (G3 < 10)
- 0 = successful (G3 >= 10)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load data

In [ ]:
data_dir = Path("data/processed")
X_full = pd.read_parquet(data_dir / "X_full.parquet")
y = pd.read_parquet(data_dir / "y_full.parquet")
y = y["G3"] if "G3" in y.columns else y.iloc[:, 0]
print(f"Features: {X_full.shape}, Target: {len(y)}")

## 3. Train/test split and classification labels

In [ ]:
Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.20, random_state=42)
yc = (y < 10).astype(int)
yctr = yc.loc[ytr.index]
ycte = yc.loc[yte.index]
print("Training:", yctr.value_counts().to_dict())
print("Test:", ycte.value_counts().to_dict())

## 4. Fit Logistic Regression

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(Xtr_f, yctr)
y_pred = clf.predict(Xte_f)
y_proba = clf.predict_proba(Xte_f)[:, 1]

print("Accuracy:", accuracy_score(ycte, y_pred))
print("Precision:", precision_score(ycte, y_pred))
print("Recall:", recall_score(ycte, y_pred))
print("F1:", f1_score(ycte, y_pred))
print("ROC_AUC:", roc_auc_score(ycte, y_proba))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(ycte, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Successful", "At-risk"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 6. Classification Report

In [ ]:
print(classification_report(ycte, y_pred, target_names=["Successful", "At-risk"]))

## 7. Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": Xtr_f.columns,
    "Coefficient": clf.named_steps["logisticregression"].coef_[0]
})
coef_df["Abs"] = coef_df["Coefficient"].abs()
display(coef_df.sort_values("Abs", ascending=False).head(10))

## 8. Classification Row

In [ ]:
tn, fp, fn, tp = cm.ravel()
classification_row = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(ycte, y_pred),
    "Precision": precision_score(ycte, y_pred),
    "Recall": recall_score(ycte, y_pred),
    "F1": f1_score(ycte, y_pred),
    "ROC_AUC": roc_auc_score(ycte, y_proba),
    "True_Negative": tn,
    "False_Positive": fp,
    "False_Negative": fn,
    "True_Positive": tp
}])
display(classification_row)

## Session 32 Complete

✅ Logistic Regression with StandardScaler
✅ Classification metrics computed
✅ Confusion matrix displayed
✅ Classification row created